## DINOCell Colab training for adipocyte segmentation in porcine brightfield images



In [2]:
import sys
print(sys.executable)

/Users/pavinsp/Documents/Cell Foundry/.venv/bin/python


In [ ]:
import torch

if torch.backends.mps.is_available():
    print("Device: MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Device: CPU")


In [2]:

from __future__ import annotations
import json
import math
import os
import random
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from huggingface_hub import hf_hub_download
from dinocell.model import DINOCell
from cellpose import plot as cellpose_plot

/usr/local/lib/python3.13/dist-packages/dinocell/dinov2/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/usr/local/lib/python3.13/dist-packages/dinocell/dinov2/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/usr/local/lib/python3.13/dist-packages/dinocell/dinov2/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [ ]:
## Local configuration (adapted from Colab)

from pathlib import Path

# CONFIG: Settings for this notebook training run; edit paths and training values here.
CONFIG = {
    "train_dir": "data/Training_porcine_bf",
    "validation_dir": "",
    "output_dir": "DINOCell/runs",
    "weights_path": ".dinocell_models/DINOCell_demo_model.pt",
    "resume_path": "",
    "steps": 1000,
    "crop_size": 512,
    "learning_rate": 2e-5,
    "weight_decay": 1e-4,
    "grad_accum_steps": 1,
    # MPS (Apple Silicon) does not support autocast the way CUDA does here;
    # run in plain fp32 for correctness over speed.
    "amp_precision": "fp32",
    "lora_rank": 8,
    "lora_alpha": 16.0,
    "lora_dropout": 0.0,
    "lora_last_blocks": 4,
    "gamma_probability": 0.15,
    "gamma_max": 1.10,
    "horizontal_flip_probability": 0.5,
    "vertical_flip_probability": 0.5,
    "validation_fraction": 0.20,
    "validate_every": 50,
    "validation_patches": 12,
    "instance_metrics": True,
    "instance_iou_threshold": 0.5,
    "save_best_checkpoint": True,
    "early_stopping": False, # changed to false
    "early_stopping_patience": 10,
    "checkpoint_every": 500,
    "positive_weight_cap": 10.0,
    "dice_weight": 0.5,
    "seed": 42,
}


In [ ]:
# Download and configure DINOCell model

MODEL_DIR = Path(".dinocell_models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_FILE = MODEL_DIR / "DINOCell_demo_model.pt"

if MODEL_FILE.exists():
    print("DINOCell model is already present:")
    print(MODEL_FILE)
else:
    print("Downloading DINOCell model from Hugging Face ...")
    downloaded = hf_hub_download(
        repo_id="KadenStillwagon/DINOCell",
        filename="DINOCell_demo_model.pt",
        local_dir=str(MODEL_DIR),
    )
    print("Download done:")
    print(downloaded)

CONFIG["weights_path"] = str(MODEL_FILE)
print("CONFIG[weights_path]:", CONFIG["weights_path"])


In [ ]:
## Training configuration

"""Standalone fine-tuning, adapted for local MPS/CPU."""

# TrainingConfig: Defines notebook configuration fields and defaults
@dataclass
class TrainingConfig:
    train_dir: str = "data/Training_porcine_bf"
    validation_dir: str = ""
    output_dir: str = "DINOCell/runs"
    weights_path: str = ".dinocell_models/DINOCell_demo_model.pt"
    resume_path: str = ""
    steps: int = 300
    crop_size: int = 512
    learning_rate: float = 2e-5
    weight_decay: float = 1e-4
    grad_accum_steps: int = 1
    amp_precision: str = "fp32"
    lora_rank: int = 8
    lora_alpha: float = 16.0
    lora_dropout: float = 0.0
    lora_last_blocks: int = 4
    gamma_probability: float = 0.15
    gamma_max: float = 1.10
    horizontal_flip_probability: float = 0.5
    vertical_flip_probability: float = 0.5
    validation_fraction: float = 0.20
    validate_every: int = 50
    validation_patches: int = 12
    instance_metrics: bool = True
    instance_iou_threshold: float = 0.5
    save_best_checkpoint: bool = False
    early_stopping: bool = False
    early_stopping_patience: int = 10
    checkpoint_every: int = 500
    positive_weight_cap: float = 10.0
    dice_weight: float = 0.5
    seed: int = 42


In [19]:
## Integration of Low-Rank Adaptation and merging functions for DINOCell

class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: float, dropout: float):
        super().__init__()
        self.base = base
        self.scale = float(alpha) / rank
        self.dropout = nn.Dropout(dropout)
        self.lora_down = nn.Linear(base.in_features, rank, bias=False).to(base.weight)
        self.lora_up = nn.Linear(rank, base.out_features, bias=False).to(base.weight)
        nn.init.kaiming_uniform_(self.lora_down.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_up.weight)
        self.base.requires_grad_(False)

    def forward(self, inputs):
        return self.base(inputs) + self.lora_up(self.lora_down(self.dropout(inputs))) * self.scale

    def merged(self):
        base = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None)
        base = base.to(self.base.weight)
        with torch.no_grad():
            base.weight.copy_(self.base.weight + (self.lora_up.weight @ self.lora_down.weight) * self.scale)
            if base.bias is not None:
                base.bias.copy_(self.base.bias)
        return base


def install_lora(model, config: TrainingConfig):
    model.requires_grad_(False)
    model.decoder.requires_grad_(True)
    model.prediction_head.requires_grad_(True)
    blocks = model.encoder.dino_encoder.blocks
    selected = range(max(0, len(blocks) - config.lora_last_blocks), len(blocks))
    names = []
    for block_index in selected:
        attention = blocks[block_index].attn
        for target in ("qkv", "proj"):
            base = getattr(attention, target)
            setattr(attention, target, LoRALinear(base, config.lora_rank, config.lora_alpha, config.lora_dropout))
            names.append(f"encoder.dino_encoder.blocks.{block_index}.attn.{target}")
    return names


def merge_lora(model):
    replacements = [(name, module) for name, module in model.named_modules() if isinstance(module, LoRALinear)]
    for name, module in replacements:
        parent_name, child_name = name.rsplit(".", 1)
        setattr(model.get_submodule(parent_name), child_name, module.merged())
    return len(replacements)


In [20]:
# Find image/mask pairs, split the data, and read files.

def find_pairs(directory: str):
    root = Path(directory).expanduser().resolve()
    supported = {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp", ".npy"}
    images, masks = {}, {}
    for path in sorted(root.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in supported:
            continue
        if path.stem.lower().endswith("_image"):
            key = str(path.relative_to(root).with_name(path.stem[:-6]))
            images[key] = path
        elif path.stem.lower().endswith("_instances"):
            key = str(path.relative_to(root).with_name(path.stem[:-10]))
            masks[key] = path
    keys = sorted(set(images) & set(masks))
    return [(key, images[key], masks[key]) for key in keys]


def split_pairs(config: TrainingConfig):
    pairs = find_pairs(config.train_dir)
    if config.validation_dir:
        return pairs, find_pairs(config.validation_dir)
    if config.validation_fraction == 0:
        return pairs, []
    indices = list(range(len(pairs)))
    random.Random(config.seed).shuffle(indices)
    count = min(len(pairs) - 1, max(1, math.ceil(len(pairs) * config.validation_fraction)))
    validation_indices = set(indices[:count])
    return ([pair for index, pair in enumerate(pairs) if index not in validation_indices],
            [pair for index, pair in enumerate(pairs) if index in validation_indices])


def read_array(path: Path):
    if path.suffix.lower() == ".npy":
        return np.load(path, allow_pickle=False)
    if path.suffix.lower() in {".tif", ".tiff"}:
        import tifffile
        return tifffile.imread(path)
    import imageio.v3 as imageio
    return imageio.imread(path)


In [8]:
# Create flows (DINOCell needs flows and masks for training)
# This module generates flow targets for instance masks using Cellpose dynamics.

def prepare_pair(image, labels):
    import cv2
    from cellpose import dynamics

    image, labels = np.asarray(image), np.asarray(labels)
    if image.shape[:2] != labels.shape[:2]:
        raise ValueError("Masks and Images have different sizes.")

    if image.ndim == 2:
        image = np.repeat(image[..., None], 3, axis=-1)
    elif image.ndim == 3 and image.shape[-1] == 1:
        image = np.repeat(image, 3, axis=-1)
    elif image.ndim == 3 and image.shape[-1] == 4:
        image = image[..., :3]

    identifiers = np.unique(np.concatenate(([0], labels.ravel())))
    labels = np.searchsorted(identifiers, labels).astype(np.int32)

    channels = []
    clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(8, 8))
    for channel in np.moveaxis(image.astype(np.float32), -1, 0):
        minimum, maximum = float(channel.min()), float(channel.max())
        normalized = ((channel - minimum) * (255.0 / (maximum - minimum))).astype(np.uint8) if maximum > minimum else np.zeros_like(channel, dtype=np.uint8)
        enhanced = clahe.apply(normalized)
        channels.append(cv2.normalize(enhanced, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8))

    image = np.stack(channels, axis=-1)

    if labels.max() == 0:
        flows = np.zeros((3, *labels.shape), dtype=np.float32)
    else:
        generated = np.asarray(
            dynamics.labels_to_flows([labels], files=None, device=torch.device("cpu"))[0],
            dtype=np.float32,
        )
        flows = generated[[3, 2, 1]]

    return {"image": image, "labels": labels, "flows": flows}


def prepare_dataset(pairs, directory: Path, prefix: str):
    directory.mkdir(parents=True, exist_ok=True)
    prepared, manifest = [], []
    started = time.perf_counter()
    for index, (key, image_path, mask_path) in enumerate(pairs):
        sample = prepare_pair(read_array(image_path), read_array(mask_path))
        path = directory / f"{prefix}_{index:04d}.npz"
        np.savez(path, **sample)
        prepared.append(path)
        manifest.append({"sample": key, "image": str(image_path), "mask": str(mask_path), "shape": list(sample["labels"].shape), "instances": int(sample["labels"].max())})
        print(f"Prepared {prefix} {index + 1}/{len(pairs)}: {key}", flush=True)
    return prepared, manifest, time.perf_counter() - started



In [ ]:
# Load and preprocess image patches for DINOCell training and evaluation.
# It also handles data augmentation (with kornia)and normalization.


def load_patch(path: Path, crop_size: int, rng: np.random.Generator):
    with np.load(path, allow_pickle=False) as data:
        image, labels, flows = data["image"], data["labels"], data["flows"]
    height, width = labels.shape
    bottom, right = max(0, crop_size - height), max(0, crop_size - width)
    image = np.pad(image, ((0, bottom), (0, right), (0, 0)), mode="edge")
    labels = np.pad(labels, ((0, bottom), (0, right)))
    flows = np.pad(flows, ((0, 0), (0, bottom), (0, right)))
    foreground = np.argwhere(labels > 0)
    if len(foreground):
        center_y, center_x = foreground[int(rng.integers(len(foreground)))]
        offset_y, offset_x = rng.integers(crop_size // 4, 3 * crop_size // 4 + 1, size=2)
        top = int(np.clip(center_y - offset_y, 0, labels.shape[0] - crop_size))
        left = int(np.clip(center_x - offset_x, 0, labels.shape[1] - crop_size))
    else:
        top = int(rng.integers(labels.shape[0] - crop_size + 1))
        left = int(rng.integers(labels.shape[1] - crop_size + 1))
    return {"image": image[top:top+crop_size, left:left+crop_size].copy(),
            "labels": labels[top:top+crop_size, left:left+crop_size].copy(),
            "flows": flows[:, top:top+crop_size, left:left+crop_size].copy()}


def build_augmentation(config: TrainingConfig, device):
    if config.gamma_probability <= 0 or config.gamma_max <= 1:
        return None
    from kornia.augmentation import RandomGamma
    return RandomGamma((1.0, config.gamma_max), p=config.gamma_probability).to(device)


def patch_tensors(sample, device, config: TrainingConfig, *, augment=False, gamma=None, rng=None):
    image = torch.from_numpy(np.ascontiguousarray(sample["image"])).permute(2, 0, 1)[None].to(device=device, dtype=torch.float32) / 255.0
    targets = torch.from_numpy(np.ascontiguousarray(sample["flows"]))[None].to(device=device, dtype=torch.float32)
    applied = []
    if augment:
        for probability, axis, channel, name in ((config.horizontal_flip_probability, -1, 0, "horizontal_flip"), (config.vertical_flip_probability, -2, 1, "vertical_flip")):
            if rng.random() < probability:
                image, targets = image.flip(axis), targets.flip(axis)
                targets[:, channel].neg_()
                applied.append(name)
        if gamma is not None:
            image = gamma(image).clamp(0, 1)
            if bool(gamma._params["batch_prob"].any().item()):
                applied.append("gamma")
    from torchvision import transforms
    image = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))(image)
    image = transforms.Resize((896, 896), interpolation=transforms.InterpolationMode.BICUBIC)(image)
    return image, targets, applied


In [22]:
# Loss functions and evaluation metrics for DINOCell model
# There are functions for computing different types of losses and evaluation metrics for the DINOCell model.


def segmentation_loss(predictions, targets, config: TrainingConfig):
    flow_dx, flow_dy, probability = [prediction.float() for prediction in predictions] # loss for flow in x, y directions and cell probability
    truth = targets[:, 2]
    probability = probability.clamp(1e-6, 1-1e-6)
    foreground = truth.sum()
    positive_weight = ((truth.numel() - foreground) / foreground.clamp_min(1)).clamp(1, config.positive_weight_cap)
    loss_bce = -(positive_weight * truth * probability.log() + (1-truth) * torch.log1p(-probability)).mean()
    loss_dice = 1 - (2 * (probability * truth).sum() + 1) / (probability.sum() + truth.sum() + 1)
    weights = torch.where(truth > 0.5, 1.0, 0.05)
    loss_dx = ((flow_dx-targets[:, 0]).square() * weights).mean()
    loss_dy = ((flow_dy-targets[:, 1]).square() * weights).mean()
    total = loss_dx + loss_dy + loss_bce + config.dice_weight * loss_dice
    return total, {"loss": total, "loss_flow_dx": loss_dx, "loss_flow_dy": loss_dy, "loss_cell_prob": loss_bce, "loss_dice": loss_dice}


def binary_metrics(predictions, targets):
    predicted, truth = predictions[2].detach() > 0.5, targets[:, 2] > 0.5
    true_positive = int((predicted & truth).sum())
    false_positive = int((predicted & ~truth).sum())
    false_negative = int((~predicted & truth).sum())
    return {"dice": 2*true_positive/max(1, 2*true_positive+false_positive+false_negative) if true_positive+false_positive+false_negative else 1.0,
            "iou": true_positive/max(1, true_positive+false_positive+false_negative) if true_positive+false_positive+false_negative else 1.0,
            "precision": true_positive/max(1, true_positive+false_positive),
            "recall": true_positive/max(1, true_positive+false_negative)}


def instance_counts(predictions, labels, config: TrainingConfig):
    from cellpose import dynamics, metrics
    flow_dx, flow_dy, probability = [prediction.detach().float().cpu().numpy()[0] for prediction in predictions]
    predicted = dynamics.compute_masks(np.stack([flow_dy, flow_dx]), probability, cellprob_threshold=0.5,
                                       flow_threshold=0.0, min_size=15, max_size_fraction=0.95, device=torch.device("cpu"))
    if isinstance(predicted, tuple):
        predicted = predicted[0]
    identifiers = np.unique(np.concatenate(([0], labels.ravel())))
    truth = np.searchsorted(identifiers, labels).astype(np.int32)
    if truth.max() == 0 or np.max(predicted) == 0:
        return 0, int(np.max(predicted)), int(truth.max())
    _, true_positive, false_positive, false_negative = metrics.average_precision(truth, predicted, threshold=[config.instance_iou_threshold])
    return int(np.ravel(true_positive)[0]), int(np.ravel(false_positive)[0]), int(np.ravel(false_negative)[0])


def evaluate(model, prepared, config, device, amp_dtype):
    was_training = model.training
    model.eval()
    rows, counts = [], np.zeros(3, dtype=np.int64)
    try:
        with torch.no_grad():
            for index in range(config.validation_patches):
                sample = load_patch(prepared[index % len(prepared)], config.crop_size, np.random.default_rng(config.seed + 10000 + index))
                image, targets, _ = patch_tensors(sample, device, config)
                with torch.autocast("cuda", dtype=amp_dtype, enabled=amp_dtype is not None):
                    predictions = model(image)
                _, components = segmentation_loss(predictions, targets, config)
                rows.append({**{key: float(value.detach()) for key, value in components.items()}, **binary_metrics(predictions, targets)})
                if config.instance_metrics:
                    counts += np.array(instance_counts(predictions, sample["labels"], config))
    finally:
        model.train(was_training)
    result = {key: float(np.mean([row[key] for row in rows])) for key in rows[0]}
    if config.instance_metrics:
        true_positive, false_positive, false_negative = [int(count) for count in counts]
        result.update(instance_precision=true_positive/max(1, true_positive+false_positive),
                      instance_recall=true_positive/max(1, true_positive+false_negative),
                      instance_f1=2*true_positive/max(1, 2*true_positive+false_positive+false_negative) if counts.sum() else 1.0,
                      instance_tp=true_positive, instance_fp=false_positive, instance_fn=false_negative)
    return result


In [ ]:
# Build the DINOCell model

def build_model(config, device):
    from dinocell.model import DINOCell

    weights = Path(config.weights_path).expanduser().resolve()
    if not weights.is_file():
        raise FileNotFoundError(f"DINOCell-Gewichte nicht gefunden: {weights}")
    model = DINOCell(dino_model=None, decoder_type="upsample", objective_type="flows", use_dino_weights=False,
                     patch_size=8, feat_size=64, crop_size=config.crop_size, drop_rate=0.05, dropout_in_encoder=True,
                     finetune_vision=False, finetune_decoder=True, finetune_prediction_head=True)
    state = torch.load(weights, map_location="cpu", weights_only=True)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model.load_state_dict(state, strict=True)
    names = install_lora(model, config)
    return model.to(device), names, str(weights)


In [12]:

# Functions for saving checkpoints and plotting training/validation curves in DINOCell
# Step changes are averaged over fixed 25-step blocks for training metrics for smoother visualization

def write_json(path, payload):
    temporary = Path(str(path) + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def save_checkpoint(path, model, optimizer, scaler, config, step, rng, metadata):
    payload = {"model_state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()},
               "optimizer_state_dict": optimizer.state_dict(), "scaler_state_dict": scaler.state_dict(),
               "config": asdict(config), "step": step, "numpy_rng_state": rng.bit_generator.state,
               "torch_rng_state": torch.get_rng_state(), "cuda_rng_states": torch.cuda.get_rng_state_all(),
               "metadata": metadata}
    temporary = Path(str(path) + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)


def plot_curves(metrics_path, directory):
    import matplotlib.pyplot as plt

    records = [
        json.loads(line)
        for line in metrics_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    directory.mkdir(parents=True, exist_ok=True)

    TRAIN_BIN = 25

    # Average training values over 25 steps.
    def binned_train(metric_name):
        points = [
            (int(row["step"]), float(row["train"][metric_name]))
            for row in records
            if metric_name in row.get("train", {})
        ]
        if not points:
            return [], []

        bins = {}
        for step, value in points:
            # 1-25 -> 25, 26-50 -> 50, ...
            bin_end = ((step - 1) // TRAIN_BIN + 1) * TRAIN_BIN
            bins.setdefault(bin_end, []).append(value)

        steps = sorted(bins)
        values = [float(np.mean(bins[step])) for step in steps]
        return steps, values

    # Read validation values without averaging.
    def validation_points(metric_name):
        points = [
            (int(row["step"]), float(row["valid"][metric_name]))
            for row in records
            if metric_name in row.get("valid", {})
        ]
        if not points:
            return [], []
        return zip(*points)

    # Training loss: mean over 25 steps.
    train_loss_names = (
        "loss",
        "loss_flow_dx",
        "loss_flow_dy",
        "loss_cell_prob",
        "loss_dice",
    )

    fig, ax = plt.subplots(figsize=(10, 4))
    plotted = False

    for name in train_loss_names:
        steps, values = binned_train(name)
        if steps:
            ax.plot(
                steps,
                values,
                linewidth=0.9,
                label=name,
            )
            plotted = True

    if plotted:
        ax.set_title("Training loss – mean over 25 steps")
        ax.set_xlabel("Training step")
        ax.set_ylabel("Loss")
        ax.grid(alpha=0.18, linewidth=0.5)
        ax.legend(fontsize=8, frameon=False)
        fig.tight_layout()
        fig.savefig(directory / "train_loss.png", dpi=160)
        plt.show()

    plt.close(fig)

    # Training Dice: mean over 25 steps.
    steps, values = binned_train("dice")

    if steps:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(
            steps,
            values,
            linewidth=1.0,
            label="Dice",
        )
        ax.set_ylim(0, 1)
        ax.set_title("Training Dice – mean over 25 steps")
        ax.set_xlabel("Training step")
        ax.set_ylabel("Dice")
        ax.grid(alpha=0.18, linewidth=0.5)
        ax.legend(fontsize=8, frameon=False)
        fig.tight_layout()
        fig.savefig(directory / "train_dice.png", dpi=160)
        plt.show()
        plt.close(fig)

    # Validation loss: one point per validation.
    valid_loss_names = (
        "loss",
        "loss_flow_dx",
        "loss_flow_dy",
        "loss_cell_prob",
        "loss_dice",
    )

    fig, ax = plt.subplots(figsize=(10, 4))
    plotted = False

    for name in valid_loss_names:
        points = [
            (int(row["step"]), float(row["valid"][name]))
            for row in records
            if name in row.get("valid", {})
        ]
        if points:
            steps, values = zip(*points)
            ax.plot(
                steps,
                values,
                linewidth=0.8,
                marker="o",
                markersize=2.0,
                label=name,
            )
            plotted = True

    if plotted:
        ax.set_title("Validation loss")
        ax.set_xlabel("Training step")
        ax.set_ylabel("Loss")
        ax.grid(alpha=0.18, linewidth=0.5)
        ax.legend(fontsize=8, frameon=False)
        fig.tight_layout()
        fig.savefig(directory / "valid_loss.png", dpi=160)
        plt.show()

    plt.close(fig)

    # Validation Dice, IoU, precision and recall.
    valid_quality_names = ("dice", "iou", "precision", "recall")

    fig, ax = plt.subplots(figsize=(10, 4))
    plotted = False

    for name in valid_quality_names:
        points = [
            (int(row["step"]), float(row["valid"][name]))
            for row in records
            if name in row.get("valid", {})
        ]
        if points:
            steps, values = zip(*points)
            linewidth = 1.1 if name == "dice" else 0.75
            markersize = 2.4 if name == "dice" else 1.8
            ax.plot(
                steps,
                values,
                linewidth=linewidth,
                marker="o",
                markersize=markersize,
                label=name,
            )
            plotted = True

    if plotted:
        ax.set_ylim(0, 1)
        ax.set_title("Validation quality")
        ax.set_xlabel("Training step")
        ax.set_ylabel("Score")
        ax.grid(alpha=0.18, linewidth=0.5)
        ax.legend(fontsize=8, frameon=False)
        fig.tight_layout()
        fig.savefig(directory / "valid_quality.png", dpi=160)
        plt.show()

    plt.close(fig)

    # Validation instance metrics.
    valid_instance_names = (
        "instance_f1",
        "instance_precision",
        "instance_recall",
    )

    fig, ax = plt.subplots(figsize=(10, 4))
    plotted = False

    for name in valid_instance_names:
        points = [
            (int(row["step"]), float(row["valid"][name]))
            for row in records
            if name in row.get("valid", {})
        ]
        if points:
            steps, values = zip(*points)
            ax.plot(
                steps,
                values,
                linewidth=0.8,
                marker="o",
                markersize=2.0,
                label=name,
            )
            plotted = True

    if plotted:
        ax.set_ylim(0, 1)
        ax.set_title("Validation instance metrics")
        ax.set_xlabel("Training step")
        ax.set_ylabel("Score")
        ax.grid(alpha=0.18, linewidth=0.5)
        ax.legend(fontsize=8, frameon=False)
        fig.tight_layout()
        fig.savefig(directory / "valid_instances.png", dpi=160)
        plt.show()

    plt.close(fig)


In [ ]:
# Main training loop for DINOCell with support for checkpointing, AMP precision, and dataset preparation
# Adapted for local MPS/CPU: no torch.cuda.* calls, autocast/GradScaler only used when actually on CUDA.

def run_training(config: TrainingConfig):
    import tempfile
    from importlib.metadata import version
    train_pairs, validation_pairs = split_pairs(config)
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    rng = np.random.default_rng(config.seed)
    device = torch.device("mps") if torch.backends.mps.is_available() else (
        torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    )
    is_cuda = device.type == "cuda"
    precision = config.amp_precision
    if precision == "bf16" and is_cuda and not torch.cuda.is_bf16_supported():
        precision = "fp16"
        print("This GPU does not support BF16; using FP16 with gradient scaling.")
    if not is_cuda and precision != "fp32":
        print(f"Requested precision '{precision}' needs CUDA; running in fp32 on {device.type} instead.")
        precision = "fp32"
    amp_dtype = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": None}[precision]
    run_dir = Path(config.output_dir).expanduser() / datetime.now(timezone.utc).strftime("dinocell_%Y%m%d_%H%M%S_%f")
    run_dir.mkdir(parents=True, exist_ok=False)
    metrics_path = run_dir / "metrics.jsonl"
    write_json(run_dir / "config.json", asdict(config))
    started = time.perf_counter()
    completed = 0
    selection = {"best_validation_dice": None, "best_step": None,
                 "validations_without_improvement": 0, "best_checkpoint": None}
    stop_reason = "steps_completed"
    metadata = {"device": device.type,
                "gpu": torch.cuda.get_device_name(0) if is_cuda else device.type,
                "amp_precision": precision,
                "selection": selection,
                "packages": {name: version(name) for name in ("dinocell", "torch", "torchvision", "cellpose", "kornia")}}
    write_json(run_dir / "status.json", {"state": "preparing", "step": 0})
    try:
        with tempfile.TemporaryDirectory(prefix="dinocell_prepared_") as working:
            training, train_manifest, train_seconds = prepare_dataset(train_pairs, Path(working), "train")
            validation, val_manifest, val_seconds = prepare_dataset(validation_pairs, Path(working), "val")
            write_json(run_dir / "dataset_manifest.json", {"train": train_manifest, "validation": val_manifest,
                                                        "preparation_seconds": train_seconds + val_seconds})
            model, lora_modules, weights = build_model(config, device)
            metadata.update(base_weights=weights, lora_modules=lora_modules,
                            total_parameters=sum(parameter.numel() for parameter in model.parameters()),
                            trainable_parameters=sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))
            print(json.dumps(metadata, indent=2), flush=True)
            optimizer = torch.optim.AdamW([parameter for parameter in model.parameters() if parameter.requires_grad],
                                         lr=config.learning_rate, weight_decay=config.weight_decay, betas=(0.9, 0.999))
            scaler = torch.amp.GradScaler("cuda", enabled=is_cuda and precision == "fp16")
            if config.resume_path:
                checkpoint = torch.load(config.resume_path, map_location="cpu", weights_only=True)
                original = checkpoint["config"]
                model.load_state_dict(checkpoint["model_state_dict"], strict=True)
                optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
                scaler.load_state_dict(checkpoint["scaler_state_dict"])
                rng.bit_generator.state = checkpoint["numpy_rng_state"]
                torch.set_rng_state(checkpoint["torch_rng_state"])
                if is_cuda and "cuda_rng_states" in checkpoint:
                    torch.cuda.set_rng_state_all(checkpoint["cuda_rng_states"])
                completed = int(checkpoint["step"])
                del checkpoint
            gamma = build_augmentation(config, device)
            model.train()
            optimizer.zero_grad(set_to_none=True)
            pending, last_saved = 0, completed
            for step in range(completed + 1, config.steps + 1):
                if is_cuda:
                    torch.cuda.synchronize()
                step_start = time.perf_counter()
                sample = load_patch(training[int(rng.integers(len(training)))], config.crop_size, rng)
                image, targets, applied = patch_tensors(sample, device, config, augment=True, gamma=gamma, rng=rng)
                with torch.autocast(device.type, dtype=amp_dtype, enabled=amp_dtype is not None and is_cuda):
                    predictions = model(image)
                loss, components = segmentation_loss(predictions, targets, config)
                scaler.scale(loss / config.grad_accum_steps).backward()
                pending += 1
                if pending == config.grad_accum_steps or step == config.steps:
                    scaler.unscale_(optimizer)
                    if pending < config.grad_accum_steps:
                        for parameter in model.parameters():
                            if parameter.grad is not None:
                                parameter.grad.mul_(config.grad_accum_steps / pending)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    pending = 0
                if is_cuda:
                    torch.cuda.synchronize()
                record = {"step": step, "train": {**{key: float(value.detach()) for key, value in components.items()},
                                                 **binary_metrics(predictions, targets)},
                          "learning_rate": optimizer.param_groups[0]["lr"], "applied_augmentations": applied,
                          "training_seconds": time.perf_counter() - step_start}
                if config.validate_every and step % config.validate_every == 0:
                    validation_start = time.perf_counter()
                    record["valid"] = evaluate(model, validation, config, device, amp_dtype if is_cuda else None)
                    record["validation_seconds"] = time.perf_counter() - validation_start
                    if config.save_best_checkpoint or config.early_stopping:
                        score = float(record["valid"]["dice"])
                        if selection["best_validation_dice"] is None or score > selection["best_validation_dice"]:
                            selection.update(best_validation_dice=score, best_step=step, validations_without_improvement=0)
                            if config.save_best_checkpoint:
                                selection["best_checkpoint"] = str((run_dir / "best.pt").resolve())
                                save_checkpoint(run_dir / "best.pt", model, optimizer, scaler, config, step, rng, metadata)
                        else:
                            selection["validations_without_improvement"] += 1
                        record["selection"] = dict(selection)
                        if config.early_stopping and selection["validations_without_improvement"] >= config.early_stopping_patience:
                            stop_reason = "early_stopping"
                with metrics_path.open("a", encoding="utf-8") as handle:
                    handle.write(json.dumps(record, allow_nan=False) + "\n")
                completed = step
                write_json(run_dir / "status.json", {"state": "training", "step": step, "steps": config.steps, "loss": record["train"]["loss"]})
                if step == 1 or step % 25 == 0 or step == config.steps:
                    print(json.dumps(record), flush=True)
                if config.checkpoint_every and step-last_saved >= config.checkpoint_every and pending == 0:
                    save_checkpoint(run_dir / "resume.pt", model, optimizer, scaler, config, completed, rng, metadata)
                    last_saved = step
                if stop_reason == "early_stopping":
                    print(f"Early stopping at step {step}: validation Dice did not improve for {config.early_stopping_patience} validations.", flush=True)
                    break
            # Resume keeps the last training state; the inference export optionally uses the best state.
            save_checkpoint(run_dir / "resume.pt", model, optimizer, scaler, config, completed, rng, metadata)
            optimizer.state.clear()
            model.to("cpu")
            export_step = completed
            if config.save_best_checkpoint and selection["best_checkpoint"]:
                best = torch.load(selection["best_checkpoint"], map_location="cpu", weights_only=True)
                model.load_state_dict(best["model_state_dict"], strict=True)
                export_step = selection["best_step"]
                del best
            merged_modules = merge_lora(model)
            destination = run_dir / ("dinocell_best_merged.pt" if config.save_best_checkpoint and selection["best_checkpoint"] else "dinocell_merged.pt")
            temporary = destination.with_suffix(".tmp")
            torch.save(model.state_dict(), temporary)
            temporary.replace(destination)
            metadata["merged_lora_modules"] = merged_modules
            del model, optimizer
            if is_cuda:
                torch.cuda.empty_cache()
        result = {"ok": True, "steps_completed": completed, "stop_reason": stop_reason, "export_step": export_step,
                  "total_seconds": time.perf_counter()-started,
                  "preparation_seconds": train_seconds+val_seconds, "train_images": len(train_pairs),
                  "validation_images": len(validation_pairs), "validation_enabled": config.validate_every > 0,
                  "best_checkpoint": selection["best_checkpoint"],
                  "merged_checkpoint": str(destination), "resume_checkpoint": str(run_dir / "resume.pt"), "metadata": metadata}
        write_json(run_dir / "result.json", result)
        write_json(run_dir / "status.json", {"state": "completed", "step": completed, "stop_reason": stop_reason})
        plot_curves(metrics_path, run_dir / "curves")
        print(f"Completed. Outputs: {run_dir}")
        return result
    except BaseException as exception:
        write_json(run_dir / "status.json", {"state": "interrupted" if isinstance(exception, KeyboardInterrupt) else "failed", "step": completed,
                                            "error": f"{type(exception).__name__}: {exception}"})
        raise


In [ ]:
# Start training with the settings from CONFIG.
import torch

config = TrainingConfig(**CONFIG)
print(f"Training: {config.steps} steps | Requested precision: {config.amp_precision}")
RUN_RESULT = run_training(config)
print(f"Finished after {RUN_RESULT['steps_completed']} steps.")
print("Model saved to:", RUN_RESULT["merged_checkpoint"])


In [ ]:
# Visualization of DINOCell predictions and predicted flows on validation patches

# Load the trained model with LoRA merged into its weights.
device = torch.device("mps") if torch.backends.mps.is_available() else (
    torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
)

model = DINOCell(
    dino_model=None,
    decoder_type="upsample",
    objective_type="flows",
    use_dino_weights=False,
    patch_size=8,
    feat_size=64,
    crop_size=512,
    drop_rate=0.05,
    dropout_in_encoder=True,
    finetune_vision=False,
    finetune_decoder=True,
    finetune_prediction_head=True,
)

trained_state = torch.load(
    RUN_RESULT["merged_checkpoint"],
    map_location="cpu",
    weights_only=True,
)

model.load_state_dict(trained_state, strict=True)
model = model.to(device)
model.eval()

print("Trained model loaded:", RUN_RESULT["merged_checkpoint"])

# Select validation patches, or training patches if no validation set is available
vis_config = TrainingConfig(**CONFIG)
train_pairs, validation_pairs = split_pairs(vis_config)

if validation_pairs:
    pairs = validation_pairs
    source_name = "Validation"
else:
    pairs = train_pairs
    source_name = "Training (no separate validation set available)"

print("Patch source:", source_name)
print("Available images:", len(pairs))

rng_vis = np.random.default_rng(123)
n_examples = 5

fig, axes = plt.subplots(
    n_examples,
    5,
    figsize=(18, 3.3 * n_examples),
    squeeze=False,
)

for row in range(n_examples):
    key, image_path, mask_path = pairs[row % len(pairs)]

    # Load the image and ground truth, then prepare Cellpose flow targets.
    sample = prepare_pair(
        read_array(image_path),
        read_array(mask_path),
    )

    temp_path = Path("/tmp") / f"dinocell_vis_{row}.npz"
    np.savez(temp_path, **sample)

    # Sample a 512x512 patch without evaluation-time augmentation.
    patch = load_patch(
        temp_path,
        vis_config.crop_size,
        rng_vis,
    )

    image_tensor, targets, _ = patch_tensors(
        patch,
        device,
        vis_config,
        augment=False,
    )

    with torch.no_grad():
        prediction = model(image_tensor)

    # Binary segmentation uses the same probability threshold as training metrics.
    pred_prob = prediction[2][0].detach().float().cpu().numpy()
    pred_mask = pred_prob > 0.5
    gt_mask = patch["labels"] > 0

    # DINOCell returns dx, dy; Cellpose visualization expects [dy, dx].
    pred_dx = prediction[0][0].detach().float().cpu().numpy()
    pred_dy = prediction[1][0].detach().float().cpu().numpy()
    flow_rgb = cellpose_plot.dx_to_circ(np.stack([pred_dy, pred_dx]))

    # Compute Dice for this patch.
    tp = np.logical_and(gt_mask, pred_mask).sum()
    fp = np.logical_and(~gt_mask, pred_mask).sum()
    fn = np.logical_and(gt_mask, ~pred_mask).sum()
    patch_dice = (2 * tp) / max(1, 2 * tp + fp + fn)

    # Normalize the image for display.
    image = patch["image"]

    if image.ndim == 2:
        display_img = image
        base_rgb = np.stack([image, image, image], axis=-1)
    elif image.ndim == 3 and image.shape[-1] in (1, 3):
        display_img = image[..., 0] if image.shape[-1] == 1 else image
        base_rgb = (
            np.stack([display_img, display_img, display_img], axis=-1)
            if np.ndim(display_img) == 2
            else display_img.copy()
        )
    elif image.ndim == 3 and image.shape[0] in (1, 3):
        display_img = np.moveaxis(image, 0, -1)
        if display_img.shape[-1] == 1:
            display_img = display_img[..., 0]
            base_rgb = np.stack([display_img, display_img, display_img], axis=-1)
        else:
            base_rgb = display_img.copy()
    else:
        display_img = image
        base_rgb = image.copy()

    base_rgb = np.asarray(base_rgb, dtype=np.float32)
    base_rgb -= base_rgb.min()
    if base_rgb.max() > 0:
        base_rgb /= base_rgb.max()

    # Overlay: green = true positive, red = false positive, blue = false negative.
    tp_mask = gt_mask & pred_mask
    fp_mask = (~gt_mask) & pred_mask
    fn_mask = gt_mask & (~pred_mask)

    overlay = base_rgb.copy()
    alpha = 0.50

    overlay[tp_mask] = (
        (1 - alpha) * overlay[tp_mask]
        + alpha * np.array([0.0, 1.0, 0.0])
    )
    overlay[fp_mask] = (
        (1 - alpha) * overlay[fp_mask]
        + alpha * np.array([1.0, 0.0, 0.0])
    )
    overlay[fn_mask] = (
        (1 - alpha) * overlay[fn_mask]
        + alpha * np.array([0.0, 0.4, 1.0])
    )

    axes[row, 0].imshow(
        display_img,
        cmap="gray" if np.ndim(display_img) == 2 else None,
    )
    axes[row, 0].set_title(f"Image\n{key}", fontsize=9)

    axes[row, 1].imshow(gt_mask, cmap="gray")
    axes[row, 1].set_title("Ground Truth", fontsize=9)

    axes[row, 2].imshow(pred_mask, cmap="gray")
    axes[row, 2].set_title(
        f"Prediction\nDice = {patch_dice:.3f}",
        fontsize=9,
    )

    axes[row, 3].imshow(flow_rgb)
    axes[row, 3].set_title(
        "Predicted Flows\nColor=direction | Brightness=strength",
        fontsize=9,
    )

    axes[row, 4].imshow(overlay)
    axes[row, 4].set_title(
        "Overlay\nGreen=TP | Red=FP | Blue=FN",
        fontsize=9,
    )

    for axis in axes[row]:
        axis.axis("off")

plt.tight_layout()
plt.show()
